<a href="https://colab.research.google.com/github/dkeitley/nano-esm/blob/data-processing/data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

"""
# Define the ESM vocabulary

This includes:

- 20 canonical amino acids
- 4 non-standard amino acids: B - Asparagine,  U - selenocysteine, Z - glutamic acid, and O - ornithine
- Beginning of sequence token: <cls>
- End of sequence token: <eos>
- Mask token: <mask>
- Padding token: <pad>
- Unknown token: <unk>

This total 29 tokens.
"""

esm_vocab_tokens = [
    "<cls>", "<pad>", "<eos>", "<unk>", "L", "A", "G", "V", "S", "E", "R",
    "T", "I", "D", "P", "K", "Q", "N", "F", "Y", "M", "H", "W", "C",
    "B", "U", "Z", "O", "<mask>"
]

# Maps each token to an integer
vocab = {token: i for i, token in enumerate(esm_vocab_tokens)}


# Reverse vocabulary for decoding
inv_vocab = {v: k for k, v in vocab.items()}


# --- 2. Custom PyTorch Dataset for Masked Language Modeling ---
# This class handles loading, tokenizing, masking, and padding the protein sequences.

class ProteinMLMDataset(Dataset):
    """
    Custom PyTorch Dataset for Masked Language Modeling on protein sequences.
    """
    def __init__(self, sequences, vocab, max_seq_len=256, mlm_prob=0.15):
        """
        Args:
            sequences (list of str): A list of protein sequences.
            vocab (dict): A dictionary mapping amino acids to integers.
            max_seq_len (int): The maximum length for padding/truncating sequences.
            mlm_prob (float): The probability of masking a token.
        """
        self.sequences = sequences
        self.vocab = vocab
        self.max_seq_len = max_seq_len
        self.mlm_prob = mlm_prob
        self.pad_token_id = self.vocab['<pad>']
        self.cls_token_id = self.vocab['<cls>']
        self.eos_token_id = self.vocab['<eos>']
        self.mask_token_id = self.vocab['<mask>']
        self.unk_token_id = self.vocab['<unk>']


    def __len__(self):
        # Returns the total number of sequences in the dataset
        return len(self.sequences)

    def __getitem__(self, idx):
        # Gets a single data point (sequence) from the dataset.
        seq = self.sequences[idx]

        # --- Tokenization ---
        tokens = [self.cls_token_id] + [self.vocab.get(aa, self.unk_token_id) for aa in seq] + [self.eos_token_id]

        # --- Masking (The Core of MLM) ---
        input_ids = list(tokens)
        labels = [-100] * len(tokens)

        for i, token in enumerate(tokens):
            # Don't mask special tokens
            if token in [self.cls_token_id, self.eos_token_id, self.pad_token_id]:
                continue

            if random.random() < self.mlm_prob:
                labels[i] = token
                rand_val = random.random()
                if rand_val < 0.8:
                    input_ids[i] = self.mask_token_id
                elif rand_val < 0.9:
                    # Choose a random token from the vocabulary (excluding special tokens)
                    first_aa_idx = self.vocab['L']
                    # The last standard amino acid in your new vocab is 'O'
                    last_aa_idx = self.vocab['O']
                    random_token = random.randint(first_aa_idx, last_aa_idx)
                    input_ids[i] = random_token
                # 10% of the time, leave the original token unchanged

        # --- Padding and Truncating ---
        input_ids = input_ids[:self.max_seq_len]
        labels = labels[:self.max_seq_len]

        padding_len = self.max_seq_len - len(input_ids)
        input_ids += [self.pad_token_id] * padding_len
        labels += [-100] * padding_len

        input_ids_tensor = torch.tensor(input_ids, dtype=torch.long)
        labels_tensor = torch.tensor(labels, dtype=torch.long)
        attention_mask = (input_ids_tensor != self.pad_token_id).long()

        return {
            'input_ids': input_ids_tensor,
            'attention_mask': attention_mask,
            'labels': labels_tensor
        }


In [18]:
# --- 3. Example Usage ---
sample_sequences = [
    "MDSKESLTPWGLILILAAMVGTGFGNVIVITLFGNDGVAKKSLQYVDQWAGKWHKMKSMS",
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK",
    "MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHQYREQIKRVKDSDDVPMVLVGNKCDLAARTVESRQAQDLARSYGIPYIETSAKTRQGVEDAFYTLVREIRQH",
    "MKSPEELKGIFEKYAAKEGDPNQLSKEELKLLLQTEFPLLK",
]

max_length = 256
dataset = ProteinMLMDataset(sample_sequences, vocab, max_seq_len=max_length)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True)

print("--- Inspecting a single batch from the DataLoader (ESM-aligned) ---")
batch = next(iter(data_loader))
input_ids = batch['input_ids']
attention_mask = batch['attention_mask']
labels = batch['labels']

print(f"Batch size: {input_ids.shape[0]}")
print(f"Sequence length: {input_ids.shape[1]}")
print("\nShape of 'input_ids':", input_ids.shape)
print("Shape of 'attention_mask':", attention_mask.shape)
print("Shape of 'labels':", labels.shape)

print("\n--- Example from the batch ---")
idx = 0
print("\nSample Input IDs (masked and padded):")
print(input_ids[idx])

print("\nCorresponding Labels (-100 for non-masked, token_id for masked):")
print(labels[idx])

decoded_input = " ".join([inv_vocab.get(token.item(), "?") for token in input_ids[idx]])
decoded_labels = " ".join([inv_vocab.get(label.item(), "_") if label.item() != -100 else "_" for label in labels[idx]])

print("\n--- Decoded Example ---")
print("Input to model: ", decoded_input)
print("Target labels : ", decoded_labels)


--- Inspecting a single batch from the DataLoader (ESM-aligned) ---
Batch size: 2
Sequence length: 256

Shape of 'input_ids': torch.Size([2, 256])
Shape of 'attention_mask': torch.Size([2, 256])
Shape of 'labels': torch.Size([2, 256])

--- Example from the batch ---

Sample Input IDs (masked and padded):
tensor([ 0, 28, 13,  8, 15, 28,  8,  4, 28, 14, 22,  6,  4, 12,  4, 28,  4,  5,
         5, 20,  7, 28, 11,  6, 18,  6, 17,  7, 12,  7, 12, 28,  4, 18,  6, 17,
        13,  6,  7,  5, 15, 15,  8,  4, 16, 28,  7, 13, 28, 22,  5, 28, 15, 28,
        21, 15, 20, 15,  8, 20,  8,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1